## Import libraries

In [1]:
import numpy as np
import pandas as pd
from obspy import read, UTCDateTime
from glob import glob
from tqdm import tqdm

## Load Hypoinverse location result

In [2]:
hypinv_catalog=pd.read_csv("/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/1.HypoInv/kim1983/Okcheon.sum", sep=',')
hypinv_catalog

,DATE TIME,SEC,LAT,LON,DEPTH,PREF,MAG,NM,NUM,GAP,...,QASR,ID-NUM,LOC,DUR,MAG1,AMP,MAG1.1,EXT,MAG,Unnamed: 23
0,2022/11/16 12:32,41.15,36.2823,127.7609,8.63,,,,55,32,...,A,200000,,,,,,,,NaN
1,2024/01/03 18:28,11.73,36.2836,127.7610,8.81,,,,69,17,...,A,200001,,,,,,,,NaN
2,2024/01/03 18:52,50.28,36.2841,127.7597,8.55,,,,35,43,...,A,200002,,,,,,,,NaN
3,2024/12/27 21:22,8.65,36.2815,127.7609,8.63,,,,68,19,...,A,200003,,,,,,,,NaN
4,2025/04/19 10:17,15.16,36.2832,127.7609,8.69,,,,50,32,...,A,200004,,,,,,,,NaN
5,2025/10/08 02:49,39.52,36.2823,127.7608,9.00,,,,77,17,...,B,200005,,,,,,,,NaN
6,2025/10/08 02:55,41.39,36.2849,127.7576,8.42,,,,12,68,...,A,200006,,,,,,,,NaN
7,2025/10/08 03:08,19.81,36.2855,127.7576,8.02,,,,10,143,...,B,200007,,,,,,,,NaN
8,2025/10/08 03:14,32.96,36.2851,127.7606,8.07,,,,10,143,...,B,200008,,,,,,,,NaN
9,2025/10/08 03:16,5.87,36.2859,127.7581,8.78,,,,23,68,...,A,200009,,,,,,,,NaN


## Modify origin times and adjust P and S picks

In [3]:
eventdir_list=sorted(glob("/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/rotated/20*"))
for i in tqdm(range(len(eventdir_list))):
    eventdir=eventdir_list[i]
    datetime=hypinv_catalog["   DATE     TIME"][i]
    second=hypinv_catalog[" SEC "][i]
    year, month, day=int(datetime[:4]), int(datetime[5:7]), int(datetime[8:10])
    hour, minute, second, microsecond=int(datetime[11:13]), int(datetime[14:16]), int(second), int((second-int(second))*1e6)
    origin_time=UTCDateTime(year, month, day, hour, minute, second, microsecond)
    sacfile_list=glob(eventdir+"/*.sac")
    
    for sacfile in sacfile_list:
        tr=read(sacfile)[0]
        
        try:
            ptime=tr.stats.starttime-tr.stats.sac.b+tr.stats.sac.a
        except:
            ptime=-9999
            
        try:
            stime=tr.stats.starttime-tr.stats.sac.b+tr.stats.sac.t0
        except:
            stime=-9999

        # Change origin time
        tr.stats.sac.nzyear=origin_time.year
        tr.stats.sac.nzjday=origin_time.julday
        tr.stats.sac.nzhour=origin_time.hour
        tr.stats.sac.nzmin=origin_time.minute
        tr.stats.sac.nzsec=origin_time.second
        tr.stats.sac.nzmsec=int(origin_time.microsecond/1000)
        
        # Adjust P/S picks
        if ptime != -9999:
            tr.stats.sac.a=ptime-origin_time
        else:
            pass
        
        if stime != -9999:
            tr.stats.sac.t0=stime-origin_time
        else:
            pass
        
        tr.write(sacfile, format="SAC")
        # print(tr.stats.sac.nzsec, tr.stats.sac.nzmsec)
        # print("\n")

100%|██████████| 12/12 [00:08<00:00,  1.43it/s]
